In [2]:
from pypdf import PdfReader

# Step 1 : Extract text from PDF



reader = PdfReader("../data/Fundamentals of Data Engineering.pdf")
full_text = ""

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    full_text += text
    print(f"Page {i+1} : {len(text)} characters extracted")

print(f"\nTotal text extracted: {len(full_text)} characters")
#print(f"\nFirst 500 characters:\n{full_text[:500]}")

Page 1 : 6 characters extracted
Page 2 : 151 characters extracted
Page 3 : 2256 characters extracted
Page 4 : 4010 characters extracted
Page 5 : 4265 characters extracted
Page 6 : 4086 characters extracted
Page 7 : 4524 characters extracted
Page 8 : 4505 characters extracted
Page 9 : 4768 characters extracted
Page 10 : 2011 characters extracted
Page 11 : 1006 characters extracted
Page 12 : 2812 characters extracted
Page 13 : 1227 characters extracted
Page 14 : 925 characters extracted
Page 15 : 2723 characters extracted
Page 16 : 2784 characters extracted
Page 17 : 2810 characters extracted
Page 18 : 1628 characters extracted
Page 19 : 2067 characters extracted
Page 20 : 2067 characters extracted
Page 21 : 1422 characters extracted
Page 22 : 2495 characters extracted
Page 23 : 2468 characters extracted
Page 24 : 2565 characters extracted
Page 25 : 100 characters extracted
Page 26 : 2222 characters extracted
Page 27 : 2519 characters extracted
Page 28 : 2482 characters extracted
Page 29

In [6]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
reader = PdfReader("../data/Fundamentals of Data Engineering.pdf")

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() or ""

print(f"Total characters: {len(full_text):,}")
print(f"Approximate words: {len(full_text.split()):,}")



Total characters: 640,978
Approximate words: 95,019


In [7]:
#Naive approach - split every N characters
naive_chunks = [full_text[i:i+500] for i in range(0, len(full_text), 500)]

print(f"\nNaive split: {len(naive_chunks)} chunks")
print(f"Problem with naive chunk #5:\n{naive_chunks[4]}")
print("\n^ Notice how it cuts mid-sentence. Context is destroyed.")


Naive split: 1282 chunks
Problem with naive chunk #5:
                                                
24 
Data Engineers and Business Leadership                                                               28 
Conclusion                                                                                                                      31 
Additional Resources                                                                                                    32  
 
2. The Data Engineering Lifecycle. . . . . . . . . . . . . . . . . . . . . . . . . 

^ Notice how it cuts mid-sentence. Context is destroyed.


In [8]:
# The smart approach - RecursiveCharacterTextSplitter
# Tries to split on paragraphs first, then sentences, then words

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, 
    chunk_overlap = 50, 
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(full_text)

print(f"Smart Split: {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")
print(f"\nSample chunk #5:\n{chunks[4]}")
print(f"\nSample chunk #6 (notice overlap with #5):\n{chunks[5]}")

print(f"\nWhy chunk_overlap=50 matters:")
print(f"Last 50 chars of chunk 5: ...{chunks[4][-50:]}")
print(f"First 50 chars of chunk 6: {chunks[5][:50]}...")

Smart Split: 1416 chunks
Average chunk size: 456 characters

Sample chunk #5:
22 Internal-Facing V ersus External-Facing Data Engineers                                     
23 Data Engineers and Other Technical Roles                                                            
24 
Data Engineers and Business Leadership                                                               28 
Conclusion                                                                                                                      31

Sample chunk #6 (notice overlap with #5):
Additional Resources                                                                                                    32  
 
2. The Data Engineering Lifecycle. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . .  33 
What Is the Data Engineering Lifecycle?                                                                    
33 The Data Lifecycle Versus the Data Engineering Lifecycle

Why chunk_ove

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a local embedding model 
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Machine learning is a subset of artificial intelligence.",
    "ML is a branch of AI that learns from data.",
    "The Eiffel Tower is in Paris.",
    "France is a country in Western Europe.",
    "Neural networks are inspired by the human brain.",
]

embeddings = model.encode(sentences)
print (f"Each sentence becomes a vector of {embeddings.shape[1]} numbers")
print (f"Shape of all embeddings: {embeddings.shape}")
print (f"\nFirst sentence vector (first 10 numbers) :")
print (embeddings[0][:10])

d:\Projects\active\rag-pdf-qa\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7745.23it/s]
BertModel LOAD REPORT from

Each sentence becomes a vector of 384 numbers
Shape of all embeddings: (5, 384)

First sentence vector (first 10 numbers) :
[-0.02345929 -0.01058116  0.0719282   0.03042549  0.02855945 -0.03882562
 -0.02819776 -0.01625061 -0.05763258 -0.00336631]


In [10]:
# Cosine similarity
def cosine_similarity(a, b):
    return np.dot(a,b) / (np.linalg.norm(a)* np.linalg.norm(b))

print ("\n Similarity scores")
query = "What is machine learning"
query_embedding = model.encode([query])[0]

for sentence, embedding in zip(sentences, embeddings):
    score = cosine_similarity(query_embedding, embedding)
    bar = "█" * int(score * 30)
    print(f"{score:.3f} {bar} {sentence[:60]}")

print(f"\nQuery: '{query}'")
print("Notice: sentences about ML score high, Paris/Europe score low.")
print("The model understands meaning, not just keywords.")


 Similarity scores
0.773 ███████████████████████ Machine learning is a subset of artificial intelligence.
0.641 ███████████████████ ML is a branch of AI that learns from data.
0.008  The Eiffel Tower is in Paris.
0.115 ███ France is a country in Western Europe.
0.358 ██████████ Neural networks are inspired by the human brain.

Query: 'What is machine learning'
Notice: sentences about ML score high, Paris/Europe score low.
The model understands meaning, not just keywords.


In [12]:
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np
import pickle

# Load and chunk PDF
reader = PdfReader("../data/Fundamentals of Data Engineering.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() or ""

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, chunk_overlap = 50
)

chunks = splitter.split_text(full_text)
print (f"Total chunks: {len(chunks)}")

# -- Embed all chunks
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
print(f"Embeddings shape: {chunk_embeddings.shape}")

# Build FAISS index - fast similarity search over millions of vectors
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))
print(f"\nFAISS index built with {index.ntotal} vectors")

# Search the index
def search (query, k=3):
    """Find the k most relevant chunks for a query"""
    query_embedding = embedding_model.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "chunk":     chunks[idx],
            "distance":  round(float(dist), 4),
            "chunk_idx": int(idx)
        })
        return results
    
# Test with real questions 

test_questions = [
        "What is machine learning?",
        "How do neural networks work?",
        "What is supervised learning?",
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    print("─" * 50)
    results = search(question, k=2)
    for i, res in enumerate(results, 1):
        print(f"Result {i} (distance: {res['distance']}):")
        print(f"{res['chunk'][:200]}...")
        print()

Total chunks: 1416


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7240.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 45/45 [00:12<00:00,  3.71it/s]

Embeddings shape: (1416, 384)

FAISS index built with 1416 vectors

Question: What is machine learning?
──────────────────────────────────────────────────
Result 1 (distance: 0.9646):
emphasizes incorporating best practices of machine learning operations (MLOps) and 
other mature practices previously adopted in software engineering and DevOps. AI 
researchers work on new, advanced ...


Question: How do neural networks work?
──────────────────────────────────────────────────
Result 1 (distance: 1.4296):
systems. 
Sources of Data: How Is Data Created? 
As you learn about the various underlying operational patterns of the systems that 
generate data, it’s essential to understand how data is created. Da...


Question: What is supervised learning?
──────────────────────────────────────────────────
Result 1 (distance: 1.2184):
ML engineers by enabling data discovery. Data lineage accelerates the time to track 
down data problems and allows consumers to locate upstream raw sources. As you 
b

In [14]:
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import faiss
import numpy as np

# ── Build the index (same as before) ──────────────────────────────────────
reader    = PdfReader("../data/Fundamentals of Data Engineering.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() or ""

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks   = splitter.split_text(full_text)

embedding_model  = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)

dimension = chunk_embeddings.shape[1]
index     = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))


# RAG prompt - instruct LLM to answer
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions 
based strictly on the provided context from a document.

Rules:
- Only use information from the context below
- If the answer is not in the context, say "I don't find this in the document"
- Be concise and precise
- Quote relevant parts when helpful

Context:
{context}"""),
    ("human", "{question}")
])

llm = ChatOllama(model = "llama3.2")
parser = StrOutputParser()
chain = rag_prompt | llm | parser

# - Full RAG function

def rag_query(question: str, k: int = 3) -> dict:
    # Step 1 - Retrieve relevant chunks
    query_embedding = embedding_model.encode([question]).astype(np.float32)
    distances, indices = index.search(query_embedding, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]

    # Step 2 - Build context from retrieved chunks
    context = "\n\n---\n\n".join(retrieved_chunks)

    # Step 3 - Generate answer grounded in context
    answer = chain.invoke({"context": context, "question": question})

    return {
        "question": question,
        "answer": answer, 
        "sources": retrieved_chunks, 
        "distances": distances[0].tolist()
    }


# ── Test with real questions 

questions = [
    "What is data engineering?",
    "Explain the key concept of monolith vs modular",
    "What are principles of good architecture?",
]

for question in questions:
    print(f"Q: {question}")
    print("─" * 60)
    result = rag_query(question)
    print(f"A: {result['answer']}")
    print(f"\nSources used ({len(result['sources'])} chunks):")
    for i, source in enumerate(result['sources'], 1):
        print(f"  [{i}] {source[:100]}...")
    print("\n" + "="*60 + "\n")





Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5184.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 45/45 [00:12<00:00,  3.62it/s]


Q: What is data engineering?
────────────────────────────────────────────────────────────
A: According to the context, "Data engineering is a set of operations aimed at creating interfaces and mechanisms for the flow and access of information." (From "Data Engineering and Its Main Concepts" by AlexSoft1)

Sources used (3 chunks):
  [1] Data engineering is a set of operations aimed at creating interfaces and mechanisms 
for the flow an...
  [2] the shadows and now sharing the stage with data science. Data engineering is one of 
the hottest fie...
  [3] and processes that take in raw data and produce high -quality, consistent information 
that supports...


Q: Explain the key concept of monolith vs modular
────────────────────────────────────────────────────────────
A: According to the context:

Monolithic systems are self-contained, often performing multiple functions under a single system.
The modular camp favors decoupled, best-of-breed technologies that perform tasks at which they ar